# Bradley-Terry Reward Model Loss（奖励模型损失）

**难度：** Medium

**函数签名：** `reward_model_loss(chosen_hidden, rejected_hidden, reward_head) -> Tensor`

**参数：**
- `chosen_hidden` — 选中响应隐藏状态 (B, D)
- `rejected_hidden` — 拒绝响应隐藏状态 (B, D)
- `reward_head` — 投影权重 (D, 1)

**返回：** 标量损失

**公式：** `loss = -mean(log(σ(r_chosen - r_rejected)))`

**约束：** 手动实现 sigmoid

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def reward_model_loss(chosen_hidden, rejected_hidden, reward_head):
    # ---- Step 1: 投影到标量奖励 ----
    # chosen_hidden @ reward_head: (B, D) @ (D, 1) → (B, 1)
    # .squeeze(-1) 去掉最后一维 → (B,)
    # 每个样本得到一个标量奖励值
    r_chosen = (chosen_hidden @ reward_head).squeeze(-1)
    r_rejected = (rejected_hidden @ reward_head).squeeze(-1)

    # ---- Step 2: 计算奖励差值 ----
    # margin = r_chosen - r_rejected
    # 理想情况: margin > 0（选中响应的奖励更高）
    margin = r_chosen - r_rejected

    # ---- Step 3: 手动实现 sigmoid + log ----
    # σ(x) = 1 / (1 + exp(-x))
    # loss = -log(σ(margin)) = -log(1/(1+exp(-margin)))
    # 当 margin >> 0: σ → 1, -log(σ) → 0（损失小，模型正确区分）
    # 当 margin << 0: σ → 0, -log(σ) → ∞（损失大，模型分不清）
    # 当 margin = 0: σ = 0.5, -log(σ) = log(2) ≈ 0.693
    loss = -torch.log(1.0 / (1.0 + torch.exp(-margin))).mean()
    return loss

In [ ]:
import math
torch.manual_seed(0)

B, D = 8, 64
reward_head = torch.randn(D, 1)

# Equal hidden states => margin = 0 => loss = -log(sigmoid(0)) = log(2)
h = torch.randn(B, D)
loss_equal = reward_model_loss(h, h, reward_head)
print(f"Equal hidden states => loss = {loss_equal.item():.4f}  (expected ≈ {math.log(2):.4f})")

# Chosen clearly better => loss should be small
chosen  = torch.ones(B, D)
rejected = -torch.ones(B, D)
loss_good = reward_model_loss(chosen, rejected, reward_head)
print(f"Chosen >> rejected  => loss = {loss_good.item():.4f}  (expected small)")

In [ ]:
from torch_judge import check
check("reward_model")

## 📝 核心思路总结

1. **Bradley-Terry 模型**：P(chosen > rejected) = σ(r_chosen - r_rejected)
2. **手动 sigmoid**：`1/(1+exp(-x))`，面试常考
3. **margin=0 时 loss=log(2)**：这是基准值，用来验证实现正确